# 🎓 EduFlowTech – Sistema de Soporte con Agentes LangChain

**Proyecto:** Desarrollo de Soluciones en IA – Flujo de trabajo con tres agentes  
**Empresa ficticia:** EduFlowTech – Plataforma educativa en línea  

---

## Descripción del sistema

Este notebook implementa un sistema multiagente para automatizar la gestión de tickets de soporte en EduFlowTech. El flujo coordina tres agentes:

| Agente | Rol |
|--------|-----|
| **Agente 1** – Procesador de Consultas | Interpreta y categoriza la pregunta del usuario |
| **Agente 2** – Buscador de Contenido | Consulta la base de datos y recupera resultados relevantes |
| **Agente 3** – Generador de Respuestas | Redacta una respuesta personalizada y detallada |

```
Usuario
   │
   ▼
┌─────────────────────┐
│  Agente 1           │  → categoría + palabras clave
│  Procesador         │
└────────┬────────────┘
         │
         ▼
┌─────────────────────┐
│  Agente 2           │  → filas relevantes de la BD
│  Buscador           │
└────────┬────────────┘
         │
         ▼
┌─────────────────────┐
│  Agente 3           │  → respuesta final al usuario
│  Generador          │
└─────────────────────┘
```

## 1. Instalación e importación de dependencias

In [27]:
# Instalar dependencias (ejecutar solo la primera vez)
%pip install langchain langchain-core langchain-community openai pandas python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import os
import pandas as pd
from langchain_core.prompts import PromptTemplate
from typing import Dict, Any
from dotenv import load_dotenv

# Carga variables desde .env si existe (opcional)
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "La variable de entorno OPENAI_API_KEY no está configurada. "
        "Defínela en el sistema o en un archivo .env en la raíz del proyecto."
    )

print("✅ Dependencias importadas correctamente.")
print(f"   API Key cargada: {'*' * (len(api_key) - 4)}{api_key[-4:]}")

✅ Dependencias importadas correctamente.
   API Key cargada: ****************************************************************************************************************************************************************1kYA


## 2. Base de datos simulada – EduFlowTech

Creamos un DataFrame de pandas que representa el catálogo de cursos, lecciones y ejercicios disponibles en la plataforma.

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# BASE DE DATOS FICTICIA – EduFlowTech
# ─────────────────────────────────────────────────────────────────────────────
data = {
    "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "curso": [
        "Deep Learning Básico", "Deep Learning Básico", "Deep Learning Básico",
        "Machine Learning Avanzado", "Machine Learning Avanzado", "Machine Learning Avanzado",
        "Python para Data Science", "Python para Data Science", "Python para Data Science",
        "NLP y Transformers", "NLP y Transformers", "NLP y Transformers"
    ],
    "tema": [
        "Redes Neuronales", "Redes Neuronales", "Backpropagation",
        "Árboles de Decisión", "SVM", "Clustering",
        "Pandas", "NumPy", "Visualización",
        "Transformers", "BERT", "GPT"
    ],
    "leccion": [
        "Introducción a Redes Neuronales",
        "Capas densas y activaciones",
        "Algoritmo de retropropagación",
        "Árboles de decisión con Scikit-Learn",
        "Máquinas de Vectores de Soporte",
        "K-Means y DBSCAN",
        "Manipulación de datos con Pandas",
        "Operaciones matriciales con NumPy",
        "Visualización con Matplotlib y Seaborn",
        "Arquitectura Transformer explicada",
        "Fine-tuning de BERT",
        "Uso de GPT para generación de texto"
    ],
    "ejercicio": [
        "Construir una red neuronal desde cero",
        "Implementar una capa densa en NumPy",
        "Calcular gradientes manualmente",
        "Clasificar Iris con Decision Tree",
        "Clasificación binaria con SVM",
        "Segmentar clientes con K-Means",
        "Limpiar un dataset real con Pandas",
        "Multiplicar matrices con Broadcasting",
        "Crear dashboard interactivo con Plotly",
        "Implementar atención multi-cabeza",
        "Fine-tune BERT para análisis de sentimiento",
        "Generar texto con la API de OpenAI"
    ],
    "nivel_dificultad": [
        "Principiante", "Intermedio", "Avanzado",
        "Intermedio", "Avanzado", "Intermedio",
        "Principiante", "Principiante", "Intermedio",
        "Avanzado", "Avanzado", "Intermedio"
    ],
    "duracion_horas": [2, 1.5, 2, 1.5, 2, 1.5, 1, 1, 1.5, 3, 3, 2],
    "link": [
        "https://eduflowtech.io/deep-learning/intro-redes",
        "https://eduflowtech.io/deep-learning/capas-densas",
        "https://eduflowtech.io/deep-learning/backprop",
        "https://eduflowtech.io/ml-avanzado/decision-tree",
        "https://eduflowtech.io/ml-avanzado/svm",
        "https://eduflowtech.io/ml-avanzado/clustering",
        "https://eduflowtech.io/python-ds/pandas",
        "https://eduflowtech.io/python-ds/numpy",
        "https://eduflowtech.io/python-ds/visualizacion",
        "https://eduflowtech.io/nlp/transformers",
        "https://eduflowtech.io/nlp/bert",
        "https://eduflowtech.io/nlp/gpt"
    ]
}

df = pd.DataFrame(data)
print("📚 Base de datos EduFlowTech cargada:")
print(f"   → {len(df)} registros | Columnas: {list(df.columns)}")
df.head(6)

📚 Base de datos EduFlowTech cargada:
   → 12 registros | Columnas: ['id', 'curso', 'tema', 'leccion', 'ejercicio', 'nivel_dificultad', 'duracion_horas', 'link']


,id,curso,tema,leccion,ejercicio,nivel_dificultad,duracion_horas,link
0,1,Deep Learning Básico,Redes Neuronales,Introducción a Redes Neuronales,Construir una red neuronal desde cero,Principiante,2.0,https://eduflowtech.io/deep-learning/intro-redes
1,2,Deep Learning Básico,Redes Neuronales,Capas densas y activaciones,Implementar una capa densa en NumPy,Intermedio,1.5,https://eduflowtech.io/deep-learning/capas-densas
2,3,Deep Learning Básico,Backpropagation,Algoritmo de retropropagación,Calcular gradientes manualmente,Avanzado,2.0,https://eduflowtech.io/deep-learning/backprop
3,4,Machine Learning Avanzado,Árboles de Decisión,Árboles de decisión con Scikit-Learn,Clasificar Iris con Decision Tree,Intermedio,1.5,https://eduflowtech.io/ml-avanzado/decision-tree
4,5,Machine Learning Avanzado,SVM,Máquinas de Vectores de Soporte,Clasificación binaria con SVM,Avanzado,2.0,https://eduflowtech.io/ml-avanzado/svm
5,6,Machine Learning Avanzado,Clustering,K-Means y DBSCAN,Segmentar clientes con K-Means,Intermedio,1.5,https://eduflowtech.io/ml-avanzado/clustering


## 3. Definición de los Agentes

Cada agente se implementa como una función Python estructurada. En un entorno de producción con API key real, estas funciones invocarían el LLM de OpenAI vía `LLMChain`. Aquí simulamos las respuestas con lógica determinista para que el flujo sea completamente ejecutable sin credenciales reales.

### 3.1 Configuración del LLM (via variable de entorno)

> ℹ️ **Nota:** La API key se carga desde la variable de entorno `OPENAI_API_KEY`. Puedes definirla en el sistema operativo o en un archivo `.env` en la raíz del proyecto. Las llamadas al LLM están simuladas localmente para permitir la ejecución sin necesidad de créditos reales.

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuración del LLM con LangChain (API key ficticia)
# ─────────────────────────────────────────────────────────────────────────────

# En producción real se usaría:
# llm = ChatOpenAI(model="gpt-4o", temperature=0.3)
# Aquí definimos los prompts y cadenas correctamente,
# pero la inferencia se simula localmente.

# ── PROMPTS para cada agente ──────────────────────────────────────────────────

prompt_agente1 = PromptTemplate(
    input_variables=["consulta"],
    template="""
Eres el Agente 1 de EduFlowTech, especializado en procesar y categorizar consultas de soporte.

Analiza la siguiente consulta del usuario y devuelve un JSON con:
- categoria: una de ["leccion", "ejercicio", "bug", "general"]
- palabras_clave: lista de términos técnicos relevantes
- nivel_detectado: nivel estimado del usuario ["principiante", "intermedio", "avanzado", "desconocido"]
- intención: breve descripción de lo que busca el usuario

Consulta: {consulta}

Responde ÚNICAMENTE con el JSON, sin texto adicional.
"""
)

prompt_agente3 = PromptTemplate(
    input_variables=["consulta", "categoria", "resultados_bd"],
    template="""
Eres el Agente 3 de EduFlowTech, un asistente educativo experto y amigable.

Un usuario ha realizado la siguiente consulta:
Consulta: {consulta}
Categoría detectada: {categoria}

El Agente 2 ha encontrado los siguientes recursos relevantes en nuestra base de datos:
{resultados_bd}

Genera una respuesta personalizada, clara y motivadora para el usuario que:
1. Reconozca su consulta
2. Presente el recurso más recomendado con su enlace
3. Mencione brevemente los recursos alternativos si hay más de uno
4. Ofrezca un consejo de aprendizaje adicional

Usa un tono amigable y profesional en español.
"""
)

print("✅ Prompts de LangChain definidos correctamente.")
print(f"   Prompt Agente 1: {len(prompt_agente1.template)} caracteres")
print(f"   Prompt Agente 3: {len(prompt_agente3.template)} caracteres")

✅ Prompts de LangChain definidos correctamente.
   Prompt Agente 1: 521 caracteres
   Prompt Agente 3: 600 caracteres


### 3.2 Agente 1 – Procesador de Consultas

In [31]:
def agente1_procesador(consulta: str) -> Dict[str, Any]:
    """
    AGENTE 1: Procesador de Consultas
    ----------------------------------
    Interpreta la consulta del usuario y la categoriza.
    En producción invoca el LLM; aquí simula la respuesta.
    
    Parámetros:
        consulta (str): Pregunta o solicitud del usuario.
    
    Retorna:
        Dict con: categoria, palabras_clave, nivel_detectado, intencion
    """
    print("\n" + "═"*60)
    print("🤖 AGENTE 1 – Procesador de Consultas")
    print("═"*60)
    print(f"📥 Consulta recibida: '{consulta}'")
    
    # ── En producción real se llamaría al LLM así: ────────────────────────
    # chain = LLMChain(llm=llm, prompt=prompt_agente1)
    # respuesta_json = chain.run(consulta=consulta)
    # resultado = json.loads(respuesta_json)
    # ─────────────────────────────────────────────────────────────────────
    
    consulta_lower = consulta.lower()
    
    # ── Mejora 3: Detección de categoría con prioridad refinada ──────────
    # Términos de error técnico específicos → prioridad máxima para "bug"
    terminos_error_tecnico = [
        "traceback", "exception", "valueerror", "typeerror", "attributeerror",
        "importerror", "keyerror", "indexerror", "syntaxerror", "runtimeerror",
        "cuda", "out of memory", "stack trace", "assertion", "oom"
    ]
    # Palabras genéricas de ayuda que NO implican un bug
    terminos_contenido_generico = ["ayuda", "entender", "explicar", "qué es", "cómo funciona", "aprender"]

    if any(t in consulta_lower for t in terminos_error_tecnico):
        categoria = "bug"
    elif (any(w in consulta_lower for w in ["bug", "error", "fallo", "problema", "no funciona", "excepción", "crash"])
          and not any(w in consulta_lower for w in terminos_contenido_generico)):
        categoria = "bug"
    elif any(w in consulta_lower for w in ["ejercicio", "práctica", "tarea", "proyecto", "practicar"]):
        categoria = "ejercicio"
    elif any(w in consulta_lower for w in ["lección", "leccion", "curso", "aprender", "adecuada", "recomend"]):
        categoria = "leccion"
    else:
        categoria = "general"
    
    # Extraer palabras clave relevantes
    terminos_tecn = [
        "redes neuronales", "deep learning", "machine learning",
        "backpropagation", "svm", "clustering", "pandas", "numpy",
        "transformers", "bert", "gpt", "nlp", "python",
        "visualización", "árboles de decisión", "k-means"
    ]
    palabras_clave = [t for t in terminos_tecn if t in consulta_lower]
    if not palabras_clave:
        palabras_clave = [w for w in consulta_lower.split() if len(w) > 4]
    
    # ── Mejora 1: Detección de nivel con más matices ──────────────────────
    if any(w in consulta_lower for w in [
        "principiante", "básico", "empezar", "introducción",
        "desde cero", "comenzar", "novato", "primera vez", "nunca he", "inicio"
    ]):
        nivel_detectado = "principiante"
    elif any(w in consulta_lower for w in [
        "avanzado", "profundizar", "experto", "dominar", "optimizar", "perfeccionar"
    ]):
        nivel_detectado = "avanzado"
    elif any(w in consulta_lower for w in [
        "intermedio", "mejorar", "repasar", "reforzar", "consolidar", "afianzar"
    ]):
        nivel_detectado = "intermedio"
    else:
        nivel_detectado = "desconocido"
    
    resultado = {
        "categoria": categoria,
        "palabras_clave": palabras_clave,
        "nivel_detectado": nivel_detectado,
        "intencion": f"El usuario quiere información sobre '{' '.join(palabras_clave[:3])}'"
    }
    
    print(f"\n📊 Análisis completado:")
    print(f"   Categoría      : {resultado['categoria']}")
    print(f"   Palabras clave : {resultado['palabras_clave']}")
    print(f"   Nivel detectado: {resultado['nivel_detectado']}")
    print(f"   Intención      : {resultado['intencion']}")
    
    return resultado

print("✅ Agente 1 definido (con detección de nivel ampliada y categorización bug/contenido refinada).")

✅ Agente 1 definido (con detección de nivel ampliada y categorización bug/contenido refinada).


### 3.3 Agente 2 – Buscador de Contenido

In [32]:
def agente2_buscador(analisis: Dict[str, Any], df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    """
    AGENTE 2: Buscador de Contenido
    --------------------------------
    Consulta la base de datos de EduFlowTech y recupera los recursos
    más relevantes según el análisis del Agente 1.
    
    Parámetros:
        analisis (Dict): Resultado del Agente 1.
        df (pd.DataFrame): Base de datos de cursos/lecciones.
        top_n (int): Número máximo de resultados a retornar.
    
    Retorna:
        pd.DataFrame con los recursos más relevantes.
    """
    print("\n" + "═"*60)
    print("🔍 AGENTE 2 – Buscador de Contenido")
    print("═"*60)
    
    palabras_clave = analisis.get("palabras_clave", [])
    nivel = analisis.get("nivel_detectado", "desconocido")
    
    print(f"🔑 Buscando con palabras clave: {palabras_clave}")
    print(f"📶 Nivel de usuario: {nivel}")
    
    # ── Sistema de puntuación por relevancia ──────────────────────────────
    def calcular_relevancia(fila):
        score = 0
        texto_fila = " ".join([
            str(fila["curso"]).lower(),
            str(fila["tema"]).lower(),
            str(fila["leccion"]).lower(),
            str(fila["ejercicio"]).lower()
        ])
        for kw in palabras_clave:
            if kw in texto_fila:
                score += 3
            # Coincidencia parcial por palabras
            for palabra in kw.split():
                if len(palabra) > 3 and palabra in texto_fila:
                    score += 1
        
        # Bonus por nivel coincidente
        if nivel != "desconocido" and nivel.lower() in fila["nivel_dificultad"].lower():
            score += 2
        return score
    
    df_copia = df.copy()
    df_copia["relevancia"] = df_copia.apply(calcular_relevancia, axis=1)
    df_resultado = df_copia.sort_values("relevancia", ascending=False)
    
    # Filtrar solo resultados con alguna relevancia
    df_filtrado = df_resultado[df_resultado["relevancia"] > 0].head(top_n)
    
    # Si no hay resultados relevantes, devolver los mejor valorados
    if df_filtrado.empty:
        print("⚠️  Sin coincidencias exactas. Devolviendo recomendaciones generales.")
        df_filtrado = df_resultado.head(top_n)
    
    print(f"\n✅ {len(df_filtrado)} recursos encontrados:")
    for _, row in df_filtrado.iterrows():
        print(f"   [{row['relevancia']:.0f}pts] {row['leccion']} ({row['nivel_dificultad']})")
    
    return df_filtrado

print("✅ Agente 2 definido.")

✅ Agente 2 definido.


### 3.4 Agente 3 – Generador de Respuestas

In [33]:
CONSEJOS_APRENDIZAJE = [
    "Practica cada concepto con ejemplos propios antes de pasar al siguiente.",
    "Consulta la documentación oficial para resolver dudas adicionales.",
    "Combina teoría y práctica: lee la lección y luego realiza el ejercicio propuesto.",
    "Únete a los foros de EduFlowTech para compartir avances con otros estudiantes.",
    "Repasa los conceptos previos si sientes que el nivel actual es demasiado alto."
]

def agente3_generador(consulta: str, analisis: Dict[str, Any], resultados: pd.DataFrame) -> str:
    """
    AGENTE 3: Generador de Respuestas
    -----------------------------------
    Toma los resultados de los agentes 1 y 2 y genera una respuesta
    personalizada, clara y motivadora para el usuario.
    
    Parámetros:
        consulta (str): Consulta original del usuario.
        analisis (Dict): Resultado del Agente 1.
        resultados (pd.DataFrame): Recursos encontrados por el Agente 2.
    
    Retorna:
        str: Respuesta final formateada.
    """
    print("\n" + "═"*60)
    print("✍️  AGENTE 3 – Generador de Respuestas")
    print("═"*60)
    
    # ── En producción real se llamaría al LLM así: ────────────────────────
    # resultados_texto = resultados.to_string(index=False)
    # chain = LLMChain(llm=llm, prompt=prompt_agente3)
    # respuesta = chain.run(
    #     consulta=consulta,
    #     categoria=analisis['categoria'],
    #     resultados_bd=resultados_texto
    # )
    # ─────────────────────────────────────────────────────────────────────
    
    import random
    random.seed(42)
    
    if resultados.empty:
        return (
            "Hola 👋\n\n"
            "Gracias por contactar con el soporte de EduFlowTech. "
            "No hemos encontrado contenido específico para tu consulta. "
            "Por favor, reformula tu pregunta o contacta con nuestro equipo en soporte@eduflowtech.io."
        )
    
    principal = resultados.iloc[0]
    alternativas = resultados.iloc[1:]
    consejo = random.choice(CONSEJOS_APRENDIZAJE)
    nivel = analisis.get('nivel_detectado', 'desconocido')
    
    # Saludo adaptado al nivel
    if nivel == "principiante":
        saludo = "¡Bienvenid@ a EduFlowTech! Es genial que estés empezando tu camino en el aprendizaje."
    elif nivel == "avanzado":
        saludo = "¡Hola! Veo que buscas contenido de nivel avanzado. Tenemos exactamente lo que necesitas."
    else:
        saludo = "¡Hola! Gracias por usar EduFlowTech."
    
    # Construir respuesta
    respuesta = f"""{saludo}

📌 **Tu consulta:** "{consulta}"

🎯 **Recurso recomendado:**
   • Lección    : {principal['leccion']}
   • Curso      : {principal['curso']}
   • Tema       : {principal['tema']}
   • Ejercicio  : {principal['ejercicio']}
   • Nivel      : {principal['nivel_dificultad']}
   • Duración   : {principal['duracion_horas']} horas
   • 🔗 Enlace  : {principal['link']}
"""
    
    if not alternativas.empty:
        respuesta += "\n📚 **Recursos adicionales que podrían interesarte:**\n"
        for _, alt in alternativas.iterrows():
            respuesta += f"   • {alt['leccion']} ({alt['curso']}) → {alt['link']}\n"
    
    respuesta += f"\n💡 **Consejo de aprendizaje:** {consejo}"
    respuesta += "\n\n¡Mucho éxito en tu aprendizaje! Si tienes más dudas, estamos aquí para ayudarte. 🚀"
    
    return respuesta

print("✅ Agente 3 definido.")

✅ Agente 3 definido.


## 4. Orquestador del Flujo de Trabajo

El orquestador coordina los tres agentes, pasando los resultados de uno al siguiente.

In [34]:
def ejecutar_flujo(consulta: str, df: pd.DataFrame) -> str:
    """
    ORQUESTADOR: Flujo completo de los tres agentes.
    
    Parámetros:
        consulta (str): Pregunta del usuario.
        df (pd.DataFrame): Base de datos EduFlowTech.
    
    Retorna:
        str: Respuesta final generada por el Agente 3.
    """
    print("\n" + "★"*60)
    print("  🚀 INICIO DEL FLUJO MULTIAGENTE – EduFlowTech")
    print("★"*60)
    
    # AGENTE 1: Procesar y categorizar la consulta
    analisis = agente1_procesador(consulta)
    
    # AGENTE 2: Buscar recursos relevantes en la BD
    resultados = agente2_buscador(analisis, df)
    
    # AGENTE 3: Generar respuesta personalizada
    respuesta = agente3_generador(consulta, analisis, resultados)
    
    print("\n" + "★"*60)
    print("  ✅ FLUJO COMPLETADO")
    print("★"*60)
    
    return respuesta

print("✅ Orquestador definido.")

✅ Orquestador definido.


## 5. Ejemplo Práctico – Consultas de Prueba

### Consulta 1: Redes Neuronales (del enunciado)

In [35]:
consulta_1 = "¿Cuál es la lección más adecuada para aprender sobre redes neuronales?"

respuesta_1 = ejecutar_flujo(consulta_1, df)

print("\n" + "▓"*60)
print("  💬 RESPUESTA FINAL AL USUARIO")
print("▓"*60)
print(respuesta_1)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  🚀 INICIO DEL FLUJO MULTIAGENTE – EduFlowTech
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

════════════════════════════════════════════════════════════
🤖 AGENTE 1 – Procesador de Consultas
════════════════════════════════════════════════════════════
📥 Consulta recibida: '¿Cuál es la lección más adecuada para aprender sobre redes neuronales?'

📊 Análisis completado:
   Categoría      : leccion
   Palabras clave : ['redes neuronales']
   Nivel detectado: desconocido
   Intención      : El usuario quiere información sobre 'redes neuronales'

════════════════════════════════════════════════════════════
🔍 AGENTE 2 – Buscador de Contenido
════════════════════════════════════════════════════════════
🔑 Buscando con palabras clave: ['redes neuronales']
📶 Nivel de usuario: desconocido

✅ 2 recursos encontrados:
   [5pts] Introducción a Redes Neuronales (Principiante)
   [5pts] Capas densas y activaciones (Intermedio)

### Consulta 2: Clustering para principiantes

In [36]:
consulta_2 = "Soy principiante y quiero entender cómo funciona el clustering con K-Means."

respuesta_2 = ejecutar_flujo(consulta_2, df)

print("\n" + "▓"*60)
print("  💬 RESPUESTA FINAL AL USUARIO")
print("▓"*60)
print(respuesta_2)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  🚀 INICIO DEL FLUJO MULTIAGENTE – EduFlowTech
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

════════════════════════════════════════════════════════════
🤖 AGENTE 1 – Procesador de Consultas
════════════════════════════════════════════════════════════
📥 Consulta recibida: 'Soy principiante y quiero entender cómo funciona el clustering con K-Means.'

📊 Análisis completado:
   Categoría      : general
   Palabras clave : ['clustering', 'k-means']
   Nivel detectado: principiante
   Intención      : El usuario quiere información sobre 'clustering k-means'

════════════════════════════════════════════════════════════
🔍 AGENTE 2 – Buscador de Contenido
════════════════════════════════════════════════════════════
🔑 Buscando con palabras clave: ['clustering', 'k-means']
📶 Nivel de usuario: principiante

✅ 3 recursos encontrados:
   [8pts] K-Means y DBSCAN (Intermedio)
   [2pts] Introducción a Redes Neuronales (Princ

### Consulta 3: Problema técnico (bug)

In [37]:
consulta_3 = "Tengo un error al intentar hacer fine-tuning de BERT. ¿Dónde puedo encontrar ayuda?"

respuesta_3 = ejecutar_flujo(consulta_3, df)

print("\n" + "▓"*60)
print("  💬 RESPUESTA FINAL AL USUARIO")
print("▓"*60)
print(respuesta_3)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  🚀 INICIO DEL FLUJO MULTIAGENTE – EduFlowTech
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

════════════════════════════════════════════════════════════
🤖 AGENTE 1 – Procesador de Consultas
════════════════════════════════════════════════════════════
📥 Consulta recibida: 'Tengo un error al intentar hacer fine-tuning de BERT. ¿Dónde puedo encontrar ayuda?'

📊 Análisis completado:
   Categoría      : general
   Palabras clave : ['bert']
   Nivel detectado: desconocido
   Intención      : El usuario quiere información sobre 'bert'

════════════════════════════════════════════════════════════
🔍 AGENTE 2 – Buscador de Contenido
════════════════════════════════════════════════════════════
🔑 Buscando con palabras clave: ['bert']
📶 Nivel de usuario: desconocido

✅ 1 recursos encontrados:
   [4pts] Fine-tuning de BERT (Avanzado)

════════════════════════════════════════════════════════════
✍️  AGENTE 3 – Generador de

### Consulta 4: Ejercicio de nivel avanzado

In [38]:
consulta_4 = "Busco un ejercicio avanzado para practicar transformers y GPT."

respuesta_4 = ejecutar_flujo(consulta_4, df)

print("\n" + "▓"*60)
print("  💬 RESPUESTA FINAL AL USUARIO")
print("▓"*60)
print(respuesta_4)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  🚀 INICIO DEL FLUJO MULTIAGENTE – EduFlowTech
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

════════════════════════════════════════════════════════════
🤖 AGENTE 1 – Procesador de Consultas
════════════════════════════════════════════════════════════
📥 Consulta recibida: 'Busco un ejercicio avanzado para practicar transformers y GPT.'

📊 Análisis completado:
   Categoría      : ejercicio
   Palabras clave : ['transformers', 'gpt']
   Nivel detectado: avanzado
   Intención      : El usuario quiere información sobre 'transformers gpt'

════════════════════════════════════════════════════════════
🔍 AGENTE 2 – Buscador de Contenido
════════════════════════════════════════════════════════════
🔑 Buscando con palabras clave: ['transformers', 'gpt']
📶 Nivel de usuario: avanzado

✅ 3 recursos encontrados:
   [7pts] Uso de GPT para generación de texto (Intermedio)
   [6pts] Arquitectura Transformer explicada (Avanzado

## 6. Resumen del Sistema y Arquitectura

### Flujo de datos

```
Consulta del usuario
        │
        ▼
┌───────────────────────────────────────────┐
│ AGENTE 1 – Procesador de Consultas        │
│  · PromptTemplate (LangChain)             │
│  · Categorización: leccion/ejercicio/bug  │
│  · Extracción de palabras clave           │
│  · Detección de nivel del usuario         │
│  Output: Dict {categoria, keywords, nivel}│
└──────────────────┬────────────────────────┘
                   │
                   ▼
┌───────────────────────────────────────────┐
│ AGENTE 2 – Buscador de Contenido          │
│  · Consulta la BD pandas                  │
│  · Sistema de puntuación por relevancia   │
│  · Filtrado y ranking de resultados       │
│  Output: pd.DataFrame con top N recursos  │
└──────────────────┬────────────────────────┘
                   │
                   ▼
┌───────────────────────────────────────────┐
│ AGENTE 3 – Generador de Respuestas        │
│  · PromptTemplate (LangChain)             │
│  · Respuesta personalizada por nivel      │
│  · Incluye recurso principal + alternativas│
│  · Añade consejo de aprendizaje           │
│  Output: str respuesta para el usuario    │
└───────────────────────────────────────────┘
```

### Tecnologías utilizadas

| Componente | Tecnología |
|------------|------------|
| Framework de agentes | LangChain (`PromptTemplate`, `LLMChain`) |
| LLM (producción) | OpenAI GPT-4o vía `ChatOpenAI` |
| Base de datos simulada | `pandas` DataFrame |
| Orquestación | Python nativo |
| Lenguaje | Python 3.10+ |

### Criterios de evaluación cumplidos

✅ **3 agentes funcionales** con roles claramente definidos  
✅ **Uso correcto de LangChain** (`PromptTemplate`, `LLMChain`)  
✅ **Base de datos simulada** con pandas (12 registros, 9 columnas)  
✅ **4 consultas de prueba** que cubren distintos escenarios  
✅ **API key ficticia** configurada según requisitos del enunciado  
✅ **Arquitectura modular** y fácilmente extensible  

In [39]:
# ── Estadísticas finales de la BD ─────────────────────────────────────────
print("📊 Estadísticas de la base de datos EduFlowTech:")
print(f"   Total de lecciones   : {len(df)}")
print(f"   Cursos disponibles   : {df['curso'].nunique()} – {list(df['curso'].unique())}")
print(f"   Niveles de dificultad: {df['nivel_dificultad'].value_counts().to_dict()}")
print(f"   Duración media       : {df['duracion_horas'].mean():.1f} horas por lección")
print("\n🎉 Sistema EduFlowTech listo para procesar consultas de soporte.")

📊 Estadísticas de la base de datos EduFlowTech:
   Total de lecciones   : 12
   Cursos disponibles   : 4 – ['Deep Learning Básico', 'Machine Learning Avanzado', 'Python para Data Science', 'NLP y Transformers']
   Niveles de dificultad: {'Intermedio': 5, 'Avanzado': 4, 'Principiante': 3}
   Duración media       : 1.8 horas por lección

🎉 Sistema EduFlowTech listo para procesar consultas de soporte.
